# backwardCGM-PD — Độ ổn định bằng bootstrapThực nghiệm **mở rộng** do nhóm bổ sung, không có trong bài báo gốc.Bài báo báo cáo độ phục hồi trung bình trên 20 replicate độc lập, tức mức chínhxác kỳ vọng trên một mẫu mới. Notebook này trả lời một câu hỏi khác: *một mô hìnhcụ thể đã chọn được thì nhạy đến mức nào với chính mẫu đã sinh ra nó?*Quy trình: chạy thuật toán một lần để có mô hình điểm, lấy `B` mẫu bootstrap**theo hàng** rồi chạy lại trên từng mẫu, sau đó đo tần suất được chọn của từngcạnh và từng phát biểu đối xứng. Chỉ lấy mẫu lại theo hàng vì cấu trúc song sinhđược xác định theo vị trí cột — mỗi hàng đã mang trọn một cặp.Hãy **Add Input** dataset `backwardCGM-PD`.

In [ ]:
import importlib.util, subprocess, sys
required = {"rdata": "rdata>=0.11", "networkx": "networkx>=3.0", "joblib": "joblib>=1.3"}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies: OK")

In [ ]:
from pathlib import Path
import json, shutil, zipfile
import pandas as pd
from IPython.display import Image, display

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/backwardCGM-PD")
RESULTS = Path("/kaggle/working/stability-results")
RESULTS.mkdir(parents=True, exist_ok=True)

# Khôi phục checkpoint từ output của một Kaggle Version trước nếu đã Add Input.
checkpoint_archives = list(INPUT_ROOT.rglob("stability-results.zip"))
if checkpoint_archives:
    with zipfile.ZipFile(checkpoint_archives[0]) as archive:
        archive.extractall(RESULTS)
    print("Restored checkpoint:", checkpoint_archives[0])

archives = list(INPUT_ROOT.rglob("backwardCGM-PD-kaggle-dataset.zip"))
scripts = list(INPUT_ROOT.rglob("python-port/experiments/simulation.py"))
if archives:
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(WORK_ROOT)
elif scripts:
    shutil.copytree(scripts[0].parents[2], WORK_ROOT, dirs_exist_ok=True)
else:
    raise FileNotFoundError("Hãy Add Input dataset backwardCGM-PD")

PORT_ROOT = WORK_ROOT / "python-port"
data_files = [
    WORK_ROOT / f"simulation/simulated-data/simdf_{s}_{p}.RData"
    for s in ("11", "22") for p in (8, 12, 16, 20)
]
missing_files = [str(p) for p in data_files if not p.exists()]
if missing_files:
    raise FileNotFoundError("Dataset Simulation không đầy đủ: " + str(missing_files))
print("Dataset: OK\nOutput:", RESULTS)

In [ ]:
# Hai thực nghiệm mở rộng do nhóm viết thêm, chưa có trong dataset gốc nên được
# nhúng thẳng vào notebook. Mã dưới đây giống hệt tệp cùng tên trong kho
# github.com/nhantrnh/DataMining.
SCRIPT = PORT_ROOT / "experiments" / "stability.py"
SCRIPT.write_text('"""Bootstrap stability of the model selected on the twin lattice.\n\nMotivation\n----------\nRoverato & Nguyen (2024) report, for every simulated configuration, the\naverage recovery of a *single* model per replicate.  Averages of that kind say\nhow often the procedure is right on a fresh sample, but they say nothing about\nhow sensitive one particular fitted model is to the sample that produced it.\nTwo configurations with identical average recovery can behave very\ndifferently: one may return nearly the same graph under resampling, the other\na different graph almost every time.\n\nThis script quantifies that missing dimension with a nonparametric bootstrap\nin the spirit of stability selection (Meinshausen & Bühlmann, 2010).  For a\ngiven dataset it\n\n1. runs the coherent backward elimination once to obtain the point estimate,\n2. draws ``B`` bootstrap resamples of the *rows* and re-runs the search on\n   each one,\n3. reports, for every edge and for every twin-symmetry statement, the fraction\n   of resamples in which it is selected.\n\nOnly rows are resampled.  The twin structure of the model is positional -\ncolumn ``j`` is paired with column ``j + p/2`` - so a row already carries a\ncomplete pair and row resampling is the correct nonparametric unit.  Permuting\nor subsetting columns would destroy the pairing the method is built on.\n\nTwo derived quantities summarise a run:\n\n``instability``\n    mean over all vertex pairs of ``2 f (1 - f)``, where ``f`` is the\n    selection frequency of the corresponding edge.  It is zero when every\n    resample returns the same edge set and reaches its maximum of 0.5 when\n    edges are selected by a coin flip.  This is the criterion of Sun, Wang &\n    Fang (2013) applied to the edge set.\n\n``exact_recovery_rate``\n    fraction of resamples returning exactly the point estimate, colour classes\n    included.\n\nUsage::\n\n    python experiments/stability.py --scenario A --p 8 --replicate 1 \\\n        --bootstrap 200 --output results/stability-A-p8.json\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nimport time\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom joblib import Parallel, delayed\n\nREPO_ROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(REPO_ROOT / "src"))\n\nfrom backward_cgm_pd.article_graphs import article_scenario_graph\nfrom backward_cgm_pd.graph import (\n    ColoredGraph,\n    Edge,\n    full_edges,\n    graph_key,\n    out_edges,\n    tau,\n)\nfrom backward_cgm_pd.io import load_simulated_datasets, write_json\nfrom backward_cgm_pd.metrics import recovery_metrics\nfrom backward_cgm_pd.search_submodel import backward_submodel\nfrom backward_cgm_pd.search_tau import backward_cgm_pd\n\nSCENARIO_CODE = {"A": "11", "B": "22"}\nSEARCHES = {"tau": backward_cgm_pd, "submodel": backward_submodel}\n\n\ndef saved_data_path(scenario: str, p: str) -> Path:\n    return (\n        REPO_ROOT\n        / "data"\n        / "simulated-data"\n        / f"simdf_{SCENARIO_CODE[scenario]}_{p}.RData"\n    )\n\n\ndef symmetric_edges(graph: ColoredGraph) -> frozenset[Edge]:\n    """Left representatives of twin edge pairs that share a single colour."""\n    return out_edges(graph.E, graph.p).TE - graph.E_atomic\n\n\ndef symmetric_vertices(graph: ColoredGraph) -> frozenset[int]:\n    """Left vertices whose diagonal element is tied to that of its twin."""\n    return frozenset(range(1, graph.p // 2 + 1)) - graph.L_atomic\n\n\ndef _fit_one(\n    data: np.ndarray,\n    indices: np.ndarray,\n    *,\n    method: str,\n    alpha: float,\n    itmax: float,\n    rcon_backend: str,\n) -> dict[str, object] | None:\n    """Fit one bootstrap resample; return ``None`` when the search fails."""\n    resample = data[indices, :]\n    search = SEARCHES[method]\n    started = time.perf_counter()\n    try:\n        result = search(\n            resample,\n            alpha=alpha,\n            itmax=int(itmax),\n            rcon_backend=rcon_backend,\n            n_jobs=1,\n        )\n    except Exception as error:  # numerically degenerate resample\n        return {"failed": True, "error": f"{type(error).__name__}: {error}"}\n    return {\n        "failed": False,\n        "runtime_seconds": time.perf_counter() - started,\n        "iterations": result.iterations,\n        "number_models": result.number_models,\n        "pvalue": result.pvalue,\n        "converged": bool(result.fit.converged),\n        "saturated": result.iterations == 0,\n        "model": result.model,\n    }\n\n\ndef bootstrap_stability(\n    data: np.ndarray,\n    *,\n    method: str,\n    bootstrap: int,\n    alpha: float,\n    itmax: int,\n    rcon_backend: str,\n    seed: int,\n    n_jobs: int,\n    verbose: bool,\n) -> dict[str, object]:\n    n, p = data.shape\n    search = SEARCHES[method]\n\n    started = time.perf_counter()\n    point = search(\n        data, alpha=alpha, itmax=itmax, rcon_backend=rcon_backend, n_jobs=1\n    )\n    point_runtime = time.perf_counter() - started\n    if verbose:\n        print(\n            f"point estimate: {point_runtime:.2f}s |E|={len(point.model.E)} "\n            f"iterations={point.iterations}",\n            flush=True,\n        )\n\n    rng = np.random.default_rng(seed)\n    all_indices = [rng.integers(0, n, n) for _ in range(bootstrap)]\n\n    outcomes = Parallel(n_jobs=n_jobs, verbose=5 if verbose else 0)(\n        delayed(_fit_one)(\n            data,\n            indices,\n            method=method,\n            alpha=alpha,\n            itmax=itmax,\n            rcon_backend=rcon_backend,\n        )\n        for indices in all_indices\n    )\n\n    successes = [row for row in outcomes if row and not row["failed"]]\n    failures = [row for row in outcomes if row and row["failed"]]\n    if not successes:\n        raise RuntimeError("every bootstrap resample failed")\n\n    universe = sorted(full_edges(p).FV)\n    left_vertices = list(range(1, p // 2 + 1))\n    left_pairs = sorted(\n        edge for edge in universe if edge[0] <= p // 2 and edge[1] <= p // 2\n    ) + sorted(\n        edge\n        for edge in universe\n        if edge[0] <= p // 2 < edge[1] and edge[1] != tau(edge[0], p)\n    )\n\n    edge_counter: Counter[Edge] = Counter()\n    symmetry_counter: Counter[Edge] = Counter()\n    vertex_counter: Counter[int] = Counter()\n    model_counter: Counter[tuple[object, ...]] = Counter()\n    for row in successes:\n        model: ColoredGraph = row["model"]  # type: ignore[assignment]\n        edge_counter.update(model.E)\n        symmetry_counter.update(symmetric_edges(model))\n        vertex_counter.update(symmetric_vertices(model))\n        model_counter[graph_key(model)] += 1\n\n    total = len(successes)\n    edge_selection = {\n        f"{i}-{j}": edge_counter[(i, j)] / total for i, j in universe\n    }\n    symmetry_selection = {\n        f"{i}-{j}": symmetry_counter[(i, j)] / total for i, j in left_pairs\n    }\n    vertex_selection = {\n        str(v): vertex_counter[v] / total for v in left_vertices\n    }\n\n    frequencies = np.array(list(edge_selection.values()))\n    instability = float(np.mean(2 * frequencies * (1 - frequencies)))\n    exact = model_counter[graph_key(point.model)] / total\n\n    agreement = [\n        recovery_metrics(row["model"], point.model).to_dict()  # type: ignore[arg-type]\n        for row in successes\n    ]\n    agreement_mean = {\n        key: float(np.nanmean([row[key] for row in agreement]))\n        for key in agreement[0]\n    }\n\n    return {\n        "point_estimate": {\n            "model": point.model,\n            "runtime_seconds": point_runtime,\n            "iterations": point.iterations,\n            "number_models": point.number_models,\n            "pvalue": point.pvalue,\n            "number_edges": len(point.model.E),\n            "number_symmetries": len(symmetric_edges(point.model)),\n            "number_parameters": point.model.n_parameters,\n        },\n        "bootstrap": {\n            "requested": bootstrap,\n            "successful": total,\n            "failed": len(failures),\n            "saturated": sum(1 for row in successes if row["saturated"]),\n            "mean_runtime": float(\n                np.mean([float(row["runtime_seconds"]) for row in successes])\n            ),\n        },\n        "instability": instability,\n        "exact_recovery_rate": exact,\n        "mean_agreement_with_point_estimate": agreement_mean,\n        "edge_selection": edge_selection,\n        "symmetry_selection": symmetry_selection,\n        "vertex_symmetry_selection": vertex_selection,\n        "distinct_models": len(model_counter),\n        "failures": [row["error"] for row in failures][:20],\n    }\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--scenario", choices=["A", "B"], required=True)\n    parser.add_argument("--p", choices=["8", "12", "16", "20"], required=True)\n    parser.add_argument(\n        "--replicate",\n        type=int,\n        default=1,\n        help="1-based index of the simulated dataset to analyse",\n    )\n    parser.add_argument("--method", choices=list(SEARCHES), default="tau")\n    parser.add_argument("--bootstrap", type=int, default=200)\n    parser.add_argument("--alpha", type=float, default=0.05)\n    parser.add_argument("--itmax", type=int, default=500)\n    parser.add_argument(\n        "--rcon-backend", choices=["mle", "grc_ipms"], default="mle"\n    )\n    parser.add_argument("--seed", type=int, default=2024)\n    parser.add_argument("--parallel", type=int, default=4)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--verbose", action="store_true")\n    args = parser.parse_args()\n\n    datasets = load_simulated_datasets(saved_data_path(args.scenario, args.p))\n    if not 1 <= args.replicate <= len(datasets):\n        parser.error(\n            f"--replicate must be within 1..{len(datasets)} for this configuration"\n        )\n    data = datasets[args.replicate - 1]\n    truth = article_scenario_graph(args.scenario, int(args.p))\n\n    result = bootstrap_stability(\n        data,\n        method=args.method,\n        bootstrap=args.bootstrap,\n        alpha=args.alpha,\n        itmax=args.itmax,\n        rcon_backend=args.rcon_backend,\n        seed=args.seed,\n        n_jobs=args.parallel,\n        verbose=args.verbose,\n    )\n    point_model = result["point_estimate"]["model"]  # type: ignore[index]\n    result["settings"] = {\n        "scenario": args.scenario,\n        "p": int(args.p),\n        "replicate": args.replicate,\n        "method": args.method,\n        "bootstrap": args.bootstrap,\n        "alpha": args.alpha,\n        "itmax": args.itmax,\n        "rcon_backend": args.rcon_backend,\n        "seed": args.seed,\n        "n_observations": int(data.shape[0]),\n    }\n    result["truth"] = truth\n    result["point_estimate_vs_truth"] = recovery_metrics(  # type: ignore[index]\n        point_model, truth\n    ).to_dict()\n\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    write_json(result, args.output)\n\n    rows = [\n        {"quantity": "edge", "label": key, "selection_frequency": value}\n        for key, value in result["edge_selection"].items()  # type: ignore[union-attr]\n    ] + [\n        {"quantity": "edge_symmetry", "label": key, "selection_frequency": value}\n        for key, value in result["symmetry_selection"].items()  # type: ignore[union-attr]\n    ] + [\n        {"quantity": "vertex_symmetry", "label": key, "selection_frequency": value}\n        for key, value in result["vertex_symmetry_selection"].items()  # type: ignore[union-attr]\n    ]\n    frame = pd.DataFrame(rows)\n    csv_path = args.output.with_name(f"{args.output.stem}-selection.csv")\n    frame.to_csv(csv_path, index=False)\n\n    print(f"Wrote {args.output}")\n    print(f"Wrote {csv_path}")\n    print(\n        f"instability={result[\'instability\']:.4f}  "\n        f"exact_recovery_rate={result[\'exact_recovery_rate\']:.3f}  "\n        f"distinct_models={result[\'distinct_models\']}  "\n        f"failed={result[\'bootstrap\'][\'failed\']}"  # type: ignore[index]\n    )\n\n\nif __name__ == "__main__":\n    main()\n')

# Script định vị dữ liệu qua REPO_ROOT/data/simulated-data, với REPO_ROOT là
# thư mục python-port; dataset Kaggle đặt chúng ở simulation/simulated-data nên
# tạo một liên kết cho khớp.
data_dir = PORT_ROOT / "data"
data_dir.mkdir(exist_ok=True)
target = data_dir / "simulated-data"
if not target.exists():
    target.symlink_to(WORK_ROOT / "simulation/simulated-data")
assert (target / "simdf_11_8.RData").exists(), "Liên kết dữ liệu không đúng"
print("Đã ghi", SCRIPT)

## Cấu hình`p = 20` tốn nhiều thời gian nhất (mỗi mẫu bootstrap mất vài phút), nên mặc địnhnotebook chạy `p = 8, 12, 16` cho cả hai kịch bản. Bật `INCLUDE_P20` nếu sessionđủ dài; kết quả đã có sẵn sẽ được bỏ qua nên có thể chạy nhiều Version nối tiếp.

In [ ]:
BOOTSTRAP = 200      # số mẫu bootstrap
REPLICATE = 1        # dùng replicate đầu tiên của mỗi cấu hình
PARALLEL = 4         # số worker; Kaggle cấp 4 vCPU
INCLUDE_P20 = False  # bật nếu muốn chạy cả p=20

ps = ["8", "12", "16"] + (["20"] if INCLUDE_P20 else [])
print({"bootstrap": BOOTSTRAP, "replicate": REPLICATE, "p": ps})

In [ ]:
import subprocess, sys, time

started = time.perf_counter()
for scenario in ("A", "B"):
    for p in ps:
        output = RESULTS / f"stability-{scenario}-p{p}-tau.json"
        if output.exists():
            print(f"Bỏ qua {scenario}/p={p}: đã có kết quả")
            continue
        command = [
            sys.executable, "-u", str(PORT_ROOT / "experiments/stability.py"),
            "--scenario", scenario, "--p", p, "--replicate", str(REPLICATE),
            "--method", "tau", "--bootstrap", str(BOOTSTRAP),
            "--parallel", str(PARALLEL), "--rcon-backend", "mle",
            "--output", str(output),
        ]
        print("Running:", " ".join(command), flush=True)
        subprocess.run(command, cwd=PORT_ROOT, check=True)
print(f"\nTổng thời gian: {time.perf_counter() - started:.1f}s")

## Tổng hợp kết quả

In [ ]:
rows = []
for path in sorted(RESULTS.glob("stability-*-tau.json")):
    d = json.loads(path.read_text())
    s, b = d["settings"], d["bootstrap"]
    rows.append({
        "scenario": s["scenario"],
        "p": s["p"],
        "instability": d["instability"],
        "exact_recovery_rate": d["exact_recovery_rate"],
        "distinct_models": d["distinct_models"],
        "point_edges": d["point_estimate"]["number_edges"],
        "point_symmetries": d["point_estimate"]["number_symmetries"],
        "bootstrap_ok": b["successful"],
        "bootstrap_failed": b["failed"],
        "mean_runtime": b["mean_runtime"],
    })
summary = pd.DataFrame(rows).sort_values(["scenario", "p"])
summary.to_csv(RESULTS / "stability-all-configurations.csv", index=False)
display(summary)

## Tần suất được chọn của từng cạnhBiểu đồ dưới đây là kết quả chính. Nếu thuật toán nhận diện được tín hiệu thật,các cạnh của đồ thị thật sẽ có tần suất gần 1 và tách hẳn khỏi phần còn lại.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

paths = sorted(RESULTS.glob("stability-*-tau.json"))
if paths:
    ncol = 2
    nrow = (len(paths) + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.5 * ncol, 3.6 * nrow), squeeze=False)
    for index, path in enumerate(paths):
        d = json.loads(path.read_text())
        ax = axes[index // ncol][index % ncol]
        freqs = np.array(sorted(d["edge_selection"].values(), reverse=True))
        truth_edges = len(d["truth"]["E"])
        colours = ["#2ca02c" if f >= 0.9 else "#d62728" if f < 0.5 else "#ff7f0e"
                   for f in freqs]
        ax.bar(range(len(freqs)), freqs, color=colours, width=0.9)
        ax.axhline(0.9, color="grey", linestyle=":", linewidth=1)
        ax.axvline(truth_edges - 0.5, color="black", linestyle="--", linewidth=1.2,
                   label=f"số cạnh thật = {truth_edges}")
        s = d["settings"]
        ax.set_title(f"Kịch bản {s['scenario']}, p={s['p']} "
                     f"(bất ổn định = {d['instability']:.3f})", fontsize=10)
        ax.set_xlabel("Cạnh, xếp theo tần suất giảm dần", fontsize=9)
        ax.set_ylabel("Tần suất được chọn", fontsize=9)
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=8, loc="upper right")
    for index in range(len(paths), nrow * ncol):
        axes[index // ncol][index % ncol].axis("off")
    fig.suptitle(f"Tần suất được chọn của từng cạnh qua {BOOTSTRAP} mẫu bootstrap")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(RESULTS / "stability-edge-frequency.png", dpi=200)
    plt.show()

archive = shutil.make_archive("/kaggle/working/stability-results", "zip", root_dir=RESULTS)
print("Download:", archive)

**Đọc kết quả:** `instability` bằng 0 khi mọi mẫu bootstrap trả về cùng một tậpcạnh, và đạt cực đại 0.5 khi mỗi cạnh được chọn như tung đồng xu.`exact_recovery_rate` là tỉ lệ mẫu trả về đúng mô hình điểm, tính cả các lớp màu.Giá trị `exact_recovery_rate` thấp không có nghĩa thuật toán kém: biểu đồ trêncho thấy các cạnh thật vẫn được chọn với tần suất gần 1, còn phần bất ổn địnhnằm ở việc mỗi mẫu thêm vào một vài cạnh nhiễu khác nhau.